In [1]:
# Load packages
import os.path as op
import glob
import shutil
import mne
import numpy as np
import pandas as pd
from mne_bids import (
    BIDSPath,
    make_dataset_description,
    print_dir_tree,
    read_raw_bids,
    write_raw_bids,
    get_anonymization_daysback,
)


In [2]:
# Make sub id list
subject_ids = np.arange(1,2)
# Define data paths
root_dir = 'C:/Users/mvmigem/Documents/data/project_2/'
raw_data_dir = op.join(root_dir,"raw")
bids_root = op.join(root_dir,"bids")
# get individual paths
sub_paths = glob.glob(raw_data_dir + "/sub*")

In [3]:
# Empty if not
if op.exists(bids_root):
    shutil.rmtree(bids_root)

In [4]:
# Event id's
event_id = {
    'start_trial':99,  
    'pos1/seq1':11, 'pos1/seq2':12, 
    'pos1/seq3':13, 'pos1/seq4':14, 
    'pos2/seq1':21, 'pos2/seq2':22,
    'pos2/seq3':23, 'pos2/seq4':24,
    'pos3/seq1':31, 'pos3/seq2':32,
    'pos3/seq3':33, 'pos3/seq4':34,
    'pos4/seq1':41, 'pos4/seq2':42,
    'pos4/seq3':43, 'pos4/seq4':44,
    }


In [5]:
raw_list = list()
bids_list = list()
# Select montage

# iterate over subjects
for i,sub in enumerate(subject_ids):
    # EEG part
    raw_path = op.join(sub_paths[i],'eeg','*.bdf')
    raw_fname = glob.glob(raw_path) [0]
    raw = mne.io.read_raw_bdf(raw_fname, preload = False)
    # specify power line frequency, and ref as required by BIDS
    raw.info['line_freq'] = 50
    ## Get events and combine with behavior data
    events = mne.find_events(raw)
    # Behaviour part
    behav_path = op.join(sub_paths[i],'behav','*.csv')
    behav_fname = glob.glob(behav_path) [0]
    behav = pd.read_csv(behav_fname)

    raw_list.append(raw)
    bids_path = BIDSPath(
        subject= f"{sub:02}",
        task = "linediscrimination",
        root = bids_root
    )

Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-01\eeg\main_01.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
4530 events found on stim channel Status
Event IDs: [   11    13    22    24    31    33    42    44    99 65536 65635 65789]


In [10]:
values_to_remove = [11,12,13,14,21,22,23,24,31,32,33,34,41,42,43,44]
filtered_events = events[np.isin(events[:, -1], values_to_remove)]

In [ ]:
lost_stim = []
shift_comp = 0
for i in range(len(filtered_events)-5):
    if filtered_events[i,2] == 99:
        if filtered_events[i +4,2] !=99:
            lost_stim.append(i+5+ shift_comp)
            shift_comp += 1
meta_data = meta_data.drop(lost_stim).reset_index(drop=True)

NameError: name 'meta_data' is not defined

In [ ]:
daysback_min, daysback_max = get_anonymization_daysback(raw_list)

In [ ]:
write_raw_bids(
    raw, bids_path, anonymize=dict(daysback=daysback_min + 2117), overwrite=True
)

In [ ]:
def validate_only_omissions(df1, df2, col_name='eeg_trigger'):
    """
    Verify that the only differences are omissions (no value mismatches)
    """
    min_length = min(len(df1), len(df2))
    
    # Compare common values (should all match if no value differences)
    common_values_match = df1[col_name].iloc[:min_length].equals(
        df2[col_name].iloc[:min_length]
    )
    
    # Check if any non-omission differences exist
    if not common_values_match:
        # Find specific value mismatches
        value_mismatches = df1[col_name].iloc[:min_length] != df2[col_name].iloc[:min_length]
        mismatch_positions = value_mismatches[value_mismatches].index.tolist()
        
        return {
            'only_omissions': False,
            'value_mismatch_positions': mismatch_positions,
            'unexpected_differences': True
        }
    
    return {
        'only_omissions': True,
        'value_mismatch_positions': [],
        'unexpected_differences': False
    }

# Usage
validation = validate_only_omissions(arr_df, behav)
print("Validation result:")
print(validation)